In [1]:
# --- Generator / Retrieval + LLM response stage ---

# 1) imports (add these to top with your existing imports)
from langchain_community.document_loaders import TextLoader, WebBaseLoader, WikipediaLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.chains.retrieval import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_community.llms import Ollama   # Ollama LLM wrapper
import os
from dotenv import load_dotenv

load_dotenv()

USER_AGENT environment variable not set, consider setting it to identify your requests.


False

In [2]:
# 1. RAG 3 Stages: Document Loading, Splitting, Embedding
text_loader = TextLoader("E:\Gen_AI\ALL_RAG_Apps\Data\VitB3.txt")
docs = text_loader.load()

web_docs = WebBaseLoader("https://www.healthline.com/nutrition/niacinamide").load()
wiki_docs = WikipediaLoader(query = 'Vitamin_B3', lang = 'en').load()

docs = docs + web_docs + wiki_docs

print(f"Number of docs before split: {len(docs)}")



ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

In [ ]:
splitted_docs = RecursiveCharacterTextSplitter(chunk_size= 500, 
                                               chunk_overlap=20).split_documents(docs)
hf_embedder = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

print(f"Number of splitted docs: {len(splitted_docs)}")

e:\Gen_AI\ALL_RAG_Apps\RAG_py3.11\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Number of splitted docs: 318


In [ ]:
# 2. Creating FAISS Vector Store DB:
faiss_db = FAISS.from_documents(splitted_docs, hf_embedder)
input_text = "What is VitB3 ?"

In [ ]:

## ----- DEBUG -----
most_matched_docs = faiss_db.similarity_search(input_text)
print(
f"""----------------------------------------------------------
The most matched docs of input query : '{input_text}' is : 
---------------------------------------------------------- 
{most_matched_docs[0].page_content}
----------------------------------------------------------"""
      )

----------------------------------------------------------
The most matched docs of input query : 'What is VitB3 ?' is : 
---------------------------------------------------------- 
Vitamin B3, colloquially referred to as niacin, is a vitamin family that includes three forms, or vitamers: nicotinic acid (niacin), nicotinamide (niacinamide), and nicotinamide riboside. All three forms of vitamin B3 are converted within the body to nicotinamide adenine dinucleotide (NAD). NAD is required for human life and people are unable to make it within their bodies without either vitamin B3 or tryptophan. Nicotinamide riboside was identified as a form of vitamin B3 in 2004.
----------------------------------------------------------


In [ ]:
# We can search the query with score. It will desplay the score also.
docs_and_score=faiss_db.similarity_search_with_score(input_text)
docs_and_score

[(Document(metadata={'title': 'Vitamin B3', 'summary': 'Vitamin B3, colloquially referred to as niacin, is a vitamin family that includes three forms, or vitamers: nicotinic acid (niacin), nicotinamide (niacinamide), and nicotinamide riboside. All three forms of vitamin B3 are converted within the body to nicotinamide adenine dinucleotide (NAD). NAD is required for human life and people are unable to make it within their bodies without either vitamin B3 or tryptophan. Nicotinamide riboside was identified as a form of vitamin B3 in 2004.\nNiacin (the nutrient) can be manufactured by plants and animals from the amino acid tryptophan. Niacin is obtained in the diet from a variety of whole and processed foods, with highest contents in fortified packaged foods, meat, poultry, red fish such as tuna and salmon, lesser amounts in nuts, legumes and seeds. Niacin as a dietary supplement is used to treat pellagra, a disease caused by niacin deficiency. Signs and symptoms of pellagra include skin 

In [ ]:
# 3. Similarity Search with input Vector
embedding_vector = hf_embedder.embed_query(input_text)
searched_docs = faiss_db.similarity_search_by_vector(embedding_vector)
searched_docs

[Document(metadata={'title': 'Vitamin B3', 'summary': 'Vitamin B3, colloquially referred to as niacin, is a vitamin family that includes three forms, or vitamers: nicotinic acid (niacin), nicotinamide (niacinamide), and nicotinamide riboside. All three forms of vitamin B3 are converted within the body to nicotinamide adenine dinucleotide (NAD). NAD is required for human life and people are unable to make it within their bodies without either vitamin B3 or tryptophan. Nicotinamide riboside was identified as a form of vitamin B3 in 2004.\nNiacin (the nutrient) can be manufactured by plants and animals from the amino acid tryptophan. Niacin is obtained in the diet from a variety of whole and processed foods, with highest contents in fortified packaged foods, meat, poultry, red fish such as tuna and salmon, lesser amounts in nuts, legumes and seeds. Niacin as a dietary supplement is used to treat pellagra, a disease caused by niacin deficiency. Signs and symptoms of pellagra include skin a

In [ ]:
# # 4. Converting FAISS DB as a Retriever
# retriever=faiss_db.as_retriever()
# docs=retriever.invoke(input_text)
# docs[0].page_content

'Vitamin B3, colloquially referred to as niacin, is a vitamin family that includes three forms, or vitamers: nicotinic acid (niacin), nicotinamide (niacinamide), and nicotinamide riboside. All three forms of vitamin B3 are converted within the body to nicotinamide adenine dinucleotide (NAD). NAD is required for human life and people are unable to make it within their bodies without either vitamin B3 or tryptophan. Nicotinamide riboside was identified as a form of vitamin B3 in 2004.'

In [ ]:
### Saving And Loading
faiss_db.save_local("faiss_DB")
new_db=FAISS.load_local("faiss_DB",
                        hf_embedder,
                        allow_dangerous_deserialization=True)

# 2) create a retriever from your FAISS DB (tweak k/search params as desired)
retriever = new_db.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [ ]:
# ----------------------------------------------------
# 3) Instantiate a FREE HuggingFace LLM for generation
# ----------------------------------------------------
print("Start")
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain.llms import HuggingFacePipeline
import torch

print("after imports")

# Choose any HF model (recommended for RAG)
model_name = "microsoft/Phi-3.5-mini-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
print("after tokenizer")

model = AutoModelForCausalLM.from_pretrained( # This line loads a pre-trained HuggingFace LLM into memory so that it can generate answers.
    model_name, 
    torch_dtype=torch.float16,
    device_map="auto"              # auto uses GPU if available, else CPU
)

print("after AutoModelForCausalLM")

gen_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.0                # deterministic like your previous Ollama config
)

print("after pipeline")

llm = HuggingFacePipeline(pipeline=gen_pipeline)

print("after HuggingFacePipeline")
# # -----------------------
# # Option A — simple RetrievalQA
# # -----------------------
from langchain.prompts import PromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain

prompt = PromptTemplate(
    input_variables=["query", "context"],
    template=(
        "You are a helpful assistant. Use the provided context to answer the user's question.\n\n"
        "Context:\n{context}\n\n"
        "Question: {query}\n\n"
        "Answer concisely and reference the context when relevant."
    )
)

# ----------------------------------------------
# 5) Document Chain
# ----------------------------------------------
document_chain = create_stuff_documents_chain(llm, prompt)
print("after create_stuff_documents_chain")

# ----------------------------------------------
# 6) Full Retrieval + Generation Chain
# ----------------------------------------------
rag_chain = create_retrieval_chain(retriever, document_chain)
print("after create_retrieval_chain")

# ----------------------------------------------
# 7) Ask a question
# ----------------------------------------------
query = "We have borne with their present government."
result = rag_chain.invoke({"question": query})

print("\n================== FINAL ANSWER ==================\n")
print(result["answer"])

print("\n================== USED DOCUMENTS ==================\n")
for i, doc in enumerate(result["context"][:3]):
    print(f"[Doc-{i}] {doc.page_content[:300]}\n")


Start


e:\Gen_AI\ALL_RAG_Apps\RAG_py3.11\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


after imports
after tokenizer


`torch_dtype` is deprecated! Use `dtype` instead!
Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

-----------------------
Notes & tips
-----------------------
- If documents are large and you want better scaling, try chain_type='map_reduce' or 'refine'.
- To tune result length, use LLM args like `max_tokens` if supported by the Ollama wrapper.
- For deterministic answers set temperature=0.0; increase for more creative responses.
- If you want to customize retrieval (e.g., use BM25-like weighting or metadata filters), set retriever.search_kwargs or pass metadata filters into .as_retriever() depending on your wrapper.